# Tutorial 1 : Generating synthetic cell type proportions


In [ ]:
#Restart runtime after every run
!git clone https://github.com/Zafar-Lab/spDDB.git

Cloning into 'spDDB'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 230 (delta 64), reused 196 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 28.38 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [ ]:
%cd spDDB/SynthST/Simulator_CTP/
!ls

/content/spDDB/SynthST/Simulator_CTP
Run_SynthST.ipynb  SynthST


**Mounting google drive for accessing the input data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install scanpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 124.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2

**Importing Libraries**

In [ ]:
#Importing Libraries
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
from sklearn.metrics.cluster import adjusted_rand_score
import tensorflow.compat.v1 as tf
from sklearn.preprocessing import normalize
tf.__version__

# the location of R (used for the mclust clustering)
os.environ['R_HOME'] = "/usr/lib/R"

import SynthST as SynthST
from SynthST.Train_SynthST import train_SynthST
from SynthST.utils import Cal_Spatial_Net, Stats_Spatial_Net, mclust_R, Cal_Spatial_Net_3D

**Initiatization**

In [ ]:
"""
The set of methods that participate in the simulation is determined by visualizing the CTPs
and selecting the ones with minimal difference from the median for all the celltypes.
The list of participating methods for all the datasets in in Supplementary Table.
Training is done on output from all the methods; this list is shuffled after each epoch.
Test data: Average/Median of all the methods (no participation in training).
"""

id  = "151508" # Dataset name
data_path = "/content/drive/MyDrive/Major_project/Benchmarking_Shared/spDDB_tutorials/1_data/"

results_ctp_path = data_path + "output_CTP/"
input_ctp_path = data_path + "input_CTP/"

methods = ["cell2location", "RCTD", "Tangram", "Stereoscope"]

**Reading datasets**

In [ ]:
adata = sc.read_visium(data_path, count_file = id + "_filtered_feature_bc_matrix.h5")
adata.obs_names_make_unique()
adata.var_names = pd.Categorical(adata.var_names).astype(str)
adata.var_names_make_unique()

def prepare_anndata(dt, adata):
    adata_X = sc.AnnData(dt)

    adata_X.obs_names = adata.obs_names
    for i in adata.uns.keys():
        adata_X.uns[i] = adata.uns[i]

    for i in adata.obs.keys():
        adata_X.obs[i] = adata.obs[i]

    for i in adata.var.keys():
        adata_X.var[i] = adata.var[i]

    for i in adata.obsm.keys():
        adata_X.obsm[i] = adata.obsm[i]

    return adata_X

**Some preprocessing on the input files**

In [ ]:
# Filter anndata based on bar_codes present in the outputs.

adata_list = []
test_data_list = {}

indices = adata.obs_names
print (len(indices))

for m in methods:
    dt = pd.read_csv(input_ctp_path + "output_" + m + id + ".csv")
    ind = dt[dt.columns[0]]
    indices = set(indices).intersection(ind)

filtered_adata = adata[adata.obs_names.isin(indices), :].copy()

print (filtered_adata.shape)
adata = filtered_adata

4384
(4382, 33538)


In [ ]:
# Find out columns/celltype with low representation
low_prop_cols = []

for m in methods:

    dt = pd.read_csv(input_ctp_path + "output_" + m + id + ".csv")
    dt = dt[dt[dt.columns[0]].isin(indices)]
    dt = dt.iloc[:,1:] # sorting will ensure that the columns are always sorted alphabetically.
    dt = dt.sort_index(axis = 1)
    cols = dt.columns
    dt = dt.to_numpy() #.transpose()
    dt = normalize(dt, axis = 1, norm = "l1")

    for col in cols:
        df = pd.DataFrame(dt, columns = cols)
        if (np.sum(df[col]) <= 10):
            if col not in low_prop_cols:
                low_prop_cols += [col]
            print (m, " method", col, "==", np.sum(df[col]), "\n")

    test_data_list[m] = dt
    adata_X = prepare_anndata(dt, adata)
    adata_list += [adata_X]

list_of_data = list(test_data_list.values())
arrays_stacked = np.stack(list_of_data)
average_array = pd.DataFrame(np.mean(arrays_stacked, axis = 0), index = adata_X.obs.index, columns = adata_X.var.index)
test_data_list["Average"] = average_array
print ("Low proportion columns", low_prop_cols)

RCTD  method L5_6_CC == 8.218973669854869 

RCTD  method Neu_mat == 6.635863456461761 

Low proportion columns ['L5_6_CC', 'Neu_mat']


**Running SynthST**

In [ ]:
tf.compat.v1.disable_eager_execution()

for adata in adata_list:

    SynthST.Cal_Spatial_Net(adata, rad_cutoff = 150)
    #SynthST.Stats_Spatial_Net(adata)

# n_epochs = 500
adata = train_SynthST(adata_list, methods + ["Average", "Median"], alpha = 0, hidden_dims = [512, 100, 10],
                     n_epochs = 200, save_reconstrction = True, save_attention = True)

------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
f1 shape is: <unknown>
f1 shape is: <unknown>
Size of Input:  (4382, 17)
100%|██████████| 200/200 [02:20<00:00,  1.42it/s]
Inference for : cell2location
[0.9999998 0.9999998 0.9999998 ... 1.0000001 1.0000001 1.0000002]
Inference for : RCTD
[0.99999976 0.99999976 0.9999998  ... 1.0000002  1.0000002  1.0000002 ]
Inference for : Tangram
[0.9999998 0.9999998 0.9999998 ... 1.0000002 1.0000002 1.0000002]
Inference for : Stereoscope
[0.9999998 0.9999998 0.9999998 ... 1.0000002 1.0000002 1.0000002]
Inference for : Average
[0.9999998 0

**Writing the output**

In [ ]:
del adata.uns["SynthST_attention"]

for i in adata.var:
    del adata.var[i]

adata.var_names = cols
print (adata.obs_names)

Index(['AAACAAGTATCTCCCA-1', 'AAACAATCTACTAGCA-1', 'AAACACCAATAACTGC-1',
       'AAACAGAGCGACTCCT-1', 'AAACAGCTTTCAGAAG-1', 'AAACAGGGTCTATATT-1',
       'AAACAGTGTTCCTGGG-1', 'AAACATTTCCCGGATT-1', 'AAACCACTACACAGAT-1',
       'AAACCCGAACGAAATC-1',
       ...
       'TTGTGTTTCCCGAAAG-1', 'TTGTTAGCAAATTCGA-1', 'TTGTTCAGTGTGCTAC-1',
       'TTGTTCTAGATACGCT-1', 'TTGTTGTGTGTCAAGA-1', 'TTGTTTCACATCCAGG-1',
       'TTGTTTCATTAGTCTA-1', 'TTGTTTCCATACAACT-1', 'TTGTTTGTATTACACG-1',
       'TTGTTTGTGTAAATTC-1'],
      dtype='object', length=4382)


In [ ]:
print (adata)
adata.write_h5ad(results_ctp_path + "simulated_st.h5ad")

AnnData object with n_obs × n_vars = 4382 × 17
    obs: 'in_tissue', 'array_row', 'array_col'
    uns: 'spatial', 'Spatial_Net'
    obsm: 'spatial', 'cell2location_embedding', 'cell2location_SynthST_ReX_Norm', 'RCTD_embedding', 'RCTD_SynthST_ReX_Norm', 'Tangram_embedding', 'Tangram_SynthST_ReX_Norm', 'Stereoscope_embedding', 'Stereoscope_SynthST_ReX_Norm', 'Average_embedding', 'Average_SynthST_ReX_Norm', 'Median_embedding', 'Median_SynthST_ReX_Norm'
